# 🎯 AI Skill Gap Analysis

This notebook demonstrates the **TF-IDF + Cosine Similarity** approach used in the CareerGuidance Skill Gap Analyzer.

**Data Source:** `data/role_requirements.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Role Requirements from CSV

In [ ]:
df = pd.read_csv('../data/role_requirements.csv')
print(f'Total rows: {len(df)}')
print(f'Roles: {df["role"].unique()}')
print(f'Types: {df["type"].unique()}')
df.head(10)

## 2. Explore Skill Distributions per Role

In [ ]:
# Count of required vs preferred skills per role
skill_counts = df.groupby(['role', 'type']).size().unstack(fill_value=0)
skill_counts.plot(kind='bar', stacked=True, color=['#2563EB', '#14B8A6'], edgecolor='white')
plt.title('Required vs Preferred Skills per Role', fontsize=14, fontweight='bold')
plt.xlabel('Role')
plt.ylabel('Number of Skills')
plt.legend(title='Skill Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Build Role Requirements Dictionary from CSV

In [ ]:
# Convert CSV to the ROLE_REQUIREMENTS dictionary structure
ROLE_REQUIREMENTS = {}
for role in df['role'].unique():
    role_df = df[df['role'] == role]
    ROLE_REQUIREMENTS[role] = {
        'required': role_df[role_df['type'] == 'required']['skill'].tolist(),
        'preferred': role_df[role_df['type'] == 'preferred']['skill'].tolist()
    }

for role, skills in ROLE_REQUIREMENTS.items():
    print(f'\n{role}:')
    print(f'  Required: {skills["required"]}')
    print(f'  Preferred: {skills["preferred"]}')

## 4. TF-IDF + Cosine Similarity Analysis

Demonstrate how a student's skills are compared against a target role using TF-IDF vectorization and cosine similarity.

In [ ]:
# Example: Student applying for Data Analyst
student_skills = 'python, sql, communication, react'
target_role = 'Data Analyst'

reqs = ROLE_REQUIREMENTS[target_role]
job_skills_str = ' '.join(reqs['required'] + reqs['preferred'])
student_skills_str = ' '.join([s.strip().lower() for s in student_skills.split(',')])

print(f'Job Skills Document: {job_skills_str}')
print(f'Student Skills Document: {student_skills_str}')

# TF-IDF Vectorization
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform([job_skills_str, student_skills_str])

print(f'\nTF-IDF Vocabulary: {vectorizer.get_feature_names_out()}')
print(f'TF-IDF Matrix Shape: {tfidf_matrix.shape}')

# Cosine Similarity
similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
match_pct = round(similarity * 100, 1)
print(f'\n✅ Cosine Similarity: {similarity:.4f}')
print(f'✅ Match Percentage: {match_pct}%')

## 5. Visualize Match Scores Across All Roles

In [ ]:
# Compare one student against ALL roles
student_skills_input = 'python, sql, machine learning, statistics, react'
student_str = ' '.join([s.strip().lower() for s in student_skills_input.split(',')])

results = []
for role, reqs in ROLE_REQUIREMENTS.items():
    job_str = ' '.join(reqs['required'] + reqs['preferred'])
    vec = TfidfVectorizer()
    mat = vec.fit_transform([job_str, student_str])
    sim = cosine_similarity(mat[0:1], mat[1:2])[0][0]
    results.append({'Role': role, 'Match %': round(sim * 100, 1)})

results_df = pd.DataFrame(results).sort_values('Match %', ascending=True)

colors = ['#EF4444' if v < 50 else '#F59E0B' if v < 75 else '#10B981' for v in results_df['Match %']]
plt.barh(results_df['Role'], results_df['Match %'], color=colors, edgecolor='white', height=0.6)
plt.xlabel('Match Percentage (%)')
plt.title(f'Skill Match Across All Roles', fontsize=14, fontweight='bold')
plt.xlim(0, 100)

for i, (_, row) in enumerate(results_df.iterrows()):
    plt.text(row['Match %'] + 1, i, f"{row['Match %']}%", va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Missing Skills Detection

In [ ]:
student_list = [s.strip().lower() for s in student_skills_input.split(',')]

for role, reqs in ROLE_REQUIREMENTS.items():
    missing = [s for s in reqs['required'] if s not in student_list]
    status = '✅ All covered' if not missing else f'❌ Missing: {", ".join(missing)}'
    print(f'{role}: {status}')